In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

train = DataLoader(datasets.FashionMNIST('.', train=True,  download=True, transform=transform), batch_size=64, shuffle=True)
test  = DataLoader(datasets.FashionMNIST('.', train=False, download=True, transform=transform), batch_size=256)

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.conv2 = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.fc    = nn.Sequential(nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 10))

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = x.flatten(1)
        x = self.fc(x)
        return x

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CNN().to(device)

In [ ]:
optimizer = optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()

for epoch in range(5):
    model.train()
    for x, y in train:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss_fn(model(x), y).backward()
        optimizer.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            predicted_classes = logits.argmax(dim=1)
            for pred, label in zip(predicted_classes, y):
                if pred == label:
                    correct += 1
                total += 1

    print(f"Epoch {epoch+1}  acc={correct / total:.3f}")